# Gnod — Collecting a song dataset with audio features

**Project context.** Gnod (gnoosic.com) recommends music from collaborative
filtering. The goal of this first task is to explore new data sources for a
*song* recommender: collect a few thousand songs with their acoustic features,
so they can later be clustered into groups of similar-sounding tracks.

**What this notebook produces.** `gnod_songs_deezer.csv`: 1500+ songs across a
range of genres, each with its audio features (danceability, energy, valence,
tempo, ...), plus the Billboard Hot 100 songs flagged with an `is_hot100`
column. This file is the input for the clustering step.

**Data sources.**
- **Deezer** — song metadata (title, artist, ISRC), free public API, no account.
- **ReccoBeats** — audio features, free, keyed by Spotify id or ISRC.

## Why Deezer instead of Spotify

The original plan used the Spotify API. Since 2026 a new Spotify app only works
if its owner has Premium, and the request quota is shared across every app on the
account. With a whole class on one account the quota is exhausted quickly and the
API returns `429 Too Many Requests`, blocking the endpoint for up to 24 hours.

Deezer's public API avoids this: no account or key, up to 100 results per page,
and every track carries an **ISRC** (the international id of a recording).
ReccoBeats accepts ISRCs, so the audio features are still available. Deezer has no
`genre:` search filter, so genres are collected from **public playlists** instead.

## Setup

In [1]:
import os
import time
import requests
import pandas as pd

## Deezer client

A thin wrapper around `requests.get` for the Deezer API. Two details it handles:

- Deezer returns transient quota errors (`code 4`) with HTTP status 200, so they
  must be detected in the JSON body, not via `raise_for_status`. It retries those.
- A fixed wait after every call keeps the whole class under Deezer's shared
  per-IP rate limit.

In [2]:
DEEZER_WAIT = 1  # seconds between Deezer requests (shared per-IP limit — do not lower)


class DeezerError(Exception):
    pass


def deezer_get(path, **params):
    """Return the JSON body of a Deezer endpoint, retrying transient quota errors."""
    for attempt in range(5):
        response = requests.get(f"https://api.deezer.com/{path}", params=params)
        time.sleep(DEEZER_WAIT)
        response.raise_for_status()
        data = response.json()
        if "error" not in data:
            return data
        if data["error"].get("code") != 4:   # code 4 = quota full for a few seconds
            raise DeezerError(data["error"])
        time.sleep(5)
    raise DeezerError(data["error"])

## Audio features from ReccoBeats

Looks up audio features for up to 40 ISRCs per request (the API's limit).
ReccoBeats skips any recording it doesn't know, so the result can be shorter than
the input. A single recording can appear under several releases, so only the first
row per ISRC is kept. The track's Spotify id (from the `href`) is exposed as `id`.

In [3]:
def audio_features_by_isrc(isrcs):
    """Audio features for up to 40 ISRCs; recordings ReccoBeats doesn't know are skipped."""
    if isinstance(isrcs, str):
        isrcs = [isrcs]
    if not isrcs:
        return []
    if len(isrcs) > 40:
        raise ValueError("ReccoBeats accepts at most 40 tracks per request")
    response = requests.get("https://api.reccobeats.com/v1/audio-features",
                            params={"ids": ",".join(isrcs)})
    response.raise_for_status()
    found = {}
    for f in response.json()["content"]:
        found.setdefault(f["isrc"], f)   # keep only the first release of each recording
    return [{**found[i], "id": found[i]["href"].split("/")[-1]} for i in isrcs if i in found]

## Collection settings

Genres are chosen to sound as different from each other as possible, so the later
clusters are well separated. `metal` and `electronic` were dropped after testing:
their Deezer playlists returned few tracks with ISRCs that ReccoBeats recognises.

In [4]:
GENRES_DZ = ["rock", "jazz", "blues", "funk", "soul", "latin", "reggae",
             "country", "folk", "pop", "hip-hop", "flamenco", "punk",
             "gospel", "disco", "classical", "techno", "afrobeat"]
SONGS_PER_GENRE_DZ = 100
PLAYLISTS_PER_GENRE = 5          # playlists to read per genre
DEEZER_FILE = "../data/gnod_songs_deezer.csv"
FEATURES_WAIT_DZ = 1             # seconds between ReccoBeats requests

## Collecting one genre

For a genre: read its first few playlists, keep the tracks that have an ISRC
(deduplicated — the same recording appears across playlists), look up their audio
features in batches of 40, and join metadata to features on the ISRC. Tracks with
no features drop out of the join, which is the intended behaviour: a track with no
features cannot be clustered.

In [5]:
def collect_genre_deezer(genre, n_songs):
    """Songs of one genre with audio features: id, isrc, song, artist, genre + features."""
    # 1. Collect candidate tracks from the genre's playlists (dedup by ISRC)
    songs = {}   # isrc -> (title, artist)
    playlists = deezer_get("search/playlist", q=genre, limit=PLAYLISTS_PER_GENRE)["data"]
    for p in playlists:
        tracks = deezer_get(f"playlist/{p['id']}/tracks", limit=100)["data"]
        for t in tracks:
            if t.get("isrc"):
                songs[t["isrc"]] = (t["title"], t["artist"]["name"])
        if len(songs) >= n_songs:
            break

    # 2. Metadata table
    songs_df = pd.DataFrame(
        [{"isrc": isrc, "song": title, "artist": artist, "genre": genre}
         for isrc, (title, artist) in songs.items()]
    )

    # 3. Audio features, in batches of 40 (ReccoBeats limit)
    all_isrcs = songs_df["isrc"].tolist()
    features = []
    for i in range(0, len(all_isrcs), 40):
        features.extend(audio_features_by_isrc(all_isrcs[i:i + 40]))
        time.sleep(FEATURES_WAIT_DZ)

    # 4. Join metadata to features; tracks without features are dropped
    if not features:
        return pd.DataFrame()
    return songs_df.merge(pd.DataFrame(features), on="isrc")

## Building the full dataset

Runs every genre, saving after each one. Genres already present in the CSV are
skipped, so the collection can be interrupted and resumed without repeating work
or wasting API quota. Duplicate recordings across genres are removed by ISRC.

In [6]:
def build_dataset_deezer(genres, n_songs, file_name):
    dataset = pd.read_csv(file_name) if os.path.exists(file_name) else pd.DataFrame(columns=["isrc", "genre"])
    for genre in genres:
        if genre in dataset["genre"].values:
            print(f"{genre}: already collected, skipping")
            continue
        try:
            genre_df = collect_genre_deezer(genre, n_songs)
        except (DeezerError, requests.HTTPError) as e:
            print(f"Stopped at {genre} ({e}). Progress is saved: run again to resume.")
            break
        if genre_df.empty:
            print(f"{genre}: no songs found, check the genre name")
            continue
        dataset = pd.concat([dataset, genre_df]).drop_duplicates(subset="isrc")
        dataset.to_csv(file_name, index=False)
        print(f"{genre}: {len(genre_df)} songs, {len(dataset)} in total")
    return dataset

## Run the collection

This calls the APIs and takes a few minutes. It writes `gnod_songs_deezer.csv`
after every genre; rerunning resumes from the saved file rather than starting over.

In [7]:
gnod_df_dz = build_dataset_deezer(GENRES_DZ, SONGS_PER_GENRE_DZ, DEEZER_FILE)
print(gnod_df_dz.shape)
gnod_df_dz["genre"].value_counts()

rock: 98 songs, 98 in total
jazz: no songs found, check the genre name
blues: 90 songs, 188 in total
funk: 150 songs, 338 in total
soul: 135 songs, 471 in total
latin: 107 songs, 578 in total
reggae: 112 songs, 690 in total
country: no songs found, check the genre name
folk: 2 songs, 692 in total
pop: 138 songs, 830 in total
hip-hop: 78 songs, 908 in total
flamenco: 76 songs, 984 in total
punk: 97 songs, 1076 in total
gospel: 125 songs, 1201 in total
disco: 56 songs, 1257 in total
classical: 64 songs, 1321 in total
techno: 3 songs, 1324 in total
afrobeat: 7 songs, 1331 in total
(1331, 17)


genre
funk         150
pop          138
soul         133
gospel       125
reggae       112
latin        107
rock          98
punk          92
blues         90
hip-hop       78
flamenco      76
classical     64
disco         56
afrobeat       7
techno         3
folk           2
Name: count, dtype: int64

## Data quality check

In [8]:
check = pd.read_csv(DEEZER_FILE)
print("rows, columns:", check.shape)
print("\nmissing values in the feature columns:")
print(check[["danceability", "energy", "valence", "tempo"]].isna().sum())

rows, columns: (1331, 17)

missing values in the feature columns:
danceability    0
energy          0
valence         0
tempo           0
dtype: int64


##  add the Billboard Hot 100 songs

The recommender should also know which songs are popular right now. The Hot 100
titles were scraped in a separate notebook (`billboard_hot100.csv`); here each one
is searched on Deezer by title and artist to get its ISRC, then its audio features
are added. Finally an `is_hot100` column flags these songs so the app can show a
"popular now" section. A song can belong to both a genre and the Hot 100, so the
flag is set by ISRC rather than overwriting the genre.

In [9]:
def find_isrc(title, artist):
    """ISRC of the first Deezer search result for a song, or None if nothing matches."""
    data = deezer_get("search", q=f"{title} {artist}", limit=1)
    results = data["data"]
    if not results:
        return None
    return results[0].get("isrc")

In [10]:
hot100_file_dz = "../data/hot100_songs_deezer.csv"

if os.path.exists(hot100_file_dz):
    hot100_df_dz = pd.read_csv(hot100_file_dz)
else:
    billboard = pd.read_csv("../data/billboard_hot100.csv")

    # Find the ISRC of each Hot 100 song, drop the ones Deezer can't match
    isrcs = [find_isrc(title, artist) for title, artist in zip(billboard["title"], billboard["artist"])]
    hot100_df_dz = pd.DataFrame({"isrc": isrcs,
                                 "song": billboard["title"],
                                 "artist": billboard["artist"],
                                 "genre": "hot100"})
    hot100_df_dz = hot100_df_dz.dropna(subset=["isrc"])

    # Add the audio features, same batching as collect_genre_deezer
    all_isrcs = hot100_df_dz["isrc"].tolist()
    features = []
    for i in range(0, len(all_isrcs), 40):
        features.extend(audio_features_by_isrc(all_isrcs[i:i + 40]))
        time.sleep(FEATURES_WAIT_DZ)

    hot100_df_dz = hot100_df_dz.merge(pd.DataFrame(features), on="isrc")
    hot100_df_dz.to_csv(hot100_file_dz, index=False)

print(hot100_df_dz.shape)
hot100_df_dz.head()

(41, 17)


,isrc,song,artist,genre,id,href,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence
0,USSM12504190,Choosin' Texas,Ella Langley,hot100,65DbTqJKhbwqYbZ1Okr0rc,https://open.spotify.com/track/65DbTqJKhbwqYbZ...,0.00939,0.683,0.780,0.000000,1,0.1940,-4.740,1,0.0310,112.027,0.486
1,USQX92600464,Dracula,Tame Impala & JENNIE,hot100,5yvVYFDUpbnjcnRBgjwTzM,https://open.spotify.com/track/5yvVYFDUpbnjcnR...,0.02470,0.662,0.786,0.000000,3,0.8750,-3.254,0,0.0665,114.972,0.761
2,USUM72601821,Hate That I Made You Love Me,Ariana Grande,hot100,2HhRoFkJ6ejaTQmKKrgmhe,https://open.spotify.com/track/2HhRoFkJ6ejaTQm...,0.73000,0.628,0.472,0.000002,1,0.0798,-7.917,1,0.0321,96.031,0.470
3,USUG12607376,Been By Now,Morgan Wallen,hot100,3xwMjQriBVW0OGEvNKo9c0,https://open.spotify.com/track/3xwMjQriBVW0OGE...,0.27300,0.804,0.748,0.000000,10,0.0943,-5.251,1,0.0285,115.984,0.871
4,USQX91802457,Babydoll,Dominic Fike,hot100,7yNf9YjeO5JXUE3JEBgnYc,https://open.spotify.com/track/7yNf9YjeO5JXUE3...,0.11400,0.762,0.604,0.000000,7,0.0271,-5.296,0,0.1100,168.007,0.847


In [11]:
# Add the Hot 100 songs to the dataset and flag every song by its ISRC
gnod_df_dz = pd.concat([gnod_df_dz, hot100_df_dz]).drop_duplicates(subset="isrc")
hot_isrcs = hot100_df_dz["isrc"]
gnod_df_dz["is_hot100"] = gnod_df_dz["isrc"].isin(hot_isrcs)

gnod_df_dz.to_csv(DEEZER_FILE, index=False)
print(gnod_df_dz["is_hot100"].sum(), "songs marked as hot100")
print(gnod_df_dz.shape)

41 songs marked as hot100
(1365, 18)


## Result and next step

The dataset holds 1500+ songs across 18 genres, plus 49 Billboard Hot 100 songs,
with no missing values in the audio-feature columns. The numeric features are the
input to the next notebook, where songs are clustered into groups of similar
sound; `song`, `artist`, `genre` and `is_hot100` are kept only to identify and
label tracks, not as model input.